# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display metadata overview
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Published: {dataset.metadata.date_published}")
print(f"Version: {dataset.metadata.version}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview

Explore available record sets and their fields by referencing their `@id` as described in the Croissant schema.

In [ ]:
# List all available record set IDs and display their fields by @id
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets are explicitly defined in the top-level metadata. Attempting to infer record sets...")
    # Try loading records directly and inspect available record set IDs
    available_record_set_ids = dataset.list_record_sets()
    print("Available Record Sets IDs:")
    for rid in available_record_set_ids:
        print(f"  - {rid}")

    # Show fields for each record set by @id
    for rid in available_record_set_ids:
        print(f"\nRecord Set '@id': {rid}")
        # Get fields for this record set
        fields = dataset.fields(record_set=rid)
        for field in fields:
            print(f"  Field '@id': {field['@id']} | name: {field.get('name','')} | dataType: {field.get('dataType','')}")
else:
    for rs in record_sets:
        print(f"Record Set '@id': {rs['@id']}")
        for field in rs.get('field', []):
            print(f"  Field '@id': {field['@id']}")

## 3. Data Extraction

Load data from desired record sets into pandas DataFrames, referencing them by their `@id`.

In [ ]:
# Get all available record set @ids
record_set_ids = dataset.list_record_sets()
print(f"Found {len(record_set_ids)} record sets:")
for rid in record_set_ids:
    print(f" - {rid}")

# Attempt to load data from each record set into a pandas DataFrame
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nLoaded {len(df)} records for record set: {rs_id}")
        print(f"Columns: {list(df.columns)}")
    else:
        print(f"No records found for record set: {rs_id}")

# Pick the first non-empty record set for further analysis
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nPrimary record set selected for EDA: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())
else:
    print("No dataframes loaded; please check the dataset schema or dataset availability.")

## 4. Exploratory Data Analysis (EDA)

Let's perform some example analyses:
  - Filter records based on a numeric field.
  - Normalize a numeric field.
  - Group data by a categorical field.

We use `@id` references for all field accesses. (Replace with actual `@id`s as appropriate.)

In [ ]:
if dataframes:
    df = dataframes[main_record_set_id]

    # Show available columns (field @ids)
    print("Available columns (field @ids):")
    print(df.columns.tolist())

    # Try to automatically pick a numeric field by checking column dtypes or names
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f"\nNumeric field selected (by @id): {numeric_field_id}")
        try:
            threshold = df[numeric_field_id].mean()
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f}:")
            display(filtered_df.head())

            # Normalize the field
            norm_col = f"{numeric_field_id}_normalized"
            filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized values of {numeric_field_id}:")
            display(filtered_df[[numeric_field_id, norm_col]].head())

            # Try to group by the first non-numeric column if possible
            group_field_id = None
            for col in df.columns:
                if col != numeric_field_id and not pd.api.types.is_numeric_dtype(df[col]):
                    group_field_id = col
                    break
            if group_field_id:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head())
        except Exception as e:
            print(f"Unable to perform EDA on numeric field: {e}")
    else:
        print("No numeric field found for EDA.")
else:
    print("No data loaded for EDA.")

## 5. Visualization

Visualize the distribution of the numeric field and its relationship with a group field (if possible), using field `@id`s for axes labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    df = dataframes[main_record_set_id]
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot, if a group field was found
    if 'group_field_id' in locals() and group_field_id:
        if group_field_id in df.columns:
            plt.figure(figsize=(10,5))
            sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
            plt.title(f"{numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(numeric_field_id)
            plt.xticks(rotation=45)
            plt.show()
else:
    print("Insufficient data or field info for visualization.")

## 6. Conclusion

- We loaded and described the FAIR² dataset using the `mlcroissant` library.
- Inspected available record sets, fields, and demonstrated structured data referencing via `@id`s.
- Loaded main record sets to pandas DataFrames and showcased basic exploratory and visualization steps.
- You can adapt the code to analyze any specific field or record set as your use case demands.